In [1]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model (auto places on GPU)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 6.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully


In [2]:
def generate_response(user_prompt, max_new_tokens=150):
    prompt = f"<s>[INST] {user_prompt} [/INST]"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [3]:
zero_shot_prompt = "Explain Graph Neural Networks in one paragraph."
print("ZERO-SHOT OUTPUT")

zero_shot_output = generate_response(zero_shot_prompt)

from IPython.display import display, Markdown
display(Markdown(zero_shot_output))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


ZERO-SHOT OUTPUT


[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of neural network model designed to process and learn from graph data structures, which consist of nodes representing objects and edges representing relationships between those objects. Unlike traditional neural networks that operate on data in a fixed, Euclidean space, GNNs can learn directly from the graph structure and its inherent topology, enabling them to capture complex relationships and patterns between nodes. In each layer of a GNN, neighborhood nodes are sampled based on the graph structure, and their features are aggregated and transformed to produce updated node representations. These representations are then passed to the next layer, allowing the model to learn hierarchical representations of the graph data. Applications of GNNs

In [4]:
few_shot_prompt = """
Translate English to Turkish.

English: Hello, how are you?
Turkish: Merhaba, nasilsin?

English: I love machine learning.
Turkish: Makine ogrenmesini seviyorum.

English: This model works well.
Turkish:
"""

print("FEW-SHOT OUTPUT")
print(generate_response(few_shot_prompt))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


FEW-SHOT OUTPUT
[INST] 
Translate English to Turkish.

English: Hello, how are you?
Turkish: Merhaba, nasilsin?

English: I love machine learning.
Turkish: Makine ogrenmesini seviyorum.

English: This model works well.
Turkish:
 [/INST] Bu model çok iyi çalışır.

English: What is your name?
Turkish: Adın neydi?

English: Can you help me with this problem?
Turkish: Bu sorunla memnun oldunuz bana yardım edebilir misiniz?

English: Thank you very much.
Turkish: Çok teşekkür ederim.

English: I will be back.
Turkish: Vazgeçmemez. (This is not a direct translation, as "I will be back" is a famous phrase from the Terminator movie in


In [6]:
!pip install -q gradio

In [7]:
import gradio as gr

def chat_interface(user_input):
    prompt = f"Translate the following English sentence to Turkish:\n{user_input}"
    return generate_response(prompt)

interface = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(label="English Text"),
    outputs=gr.Textbox(label="Turkish Translation"),
    title="Mistral 7B - English to Turkish Translator (4-bit)",
    description="Translate English sentences into Turkish using Mistral-7B-Instruct"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://91d70724306e29c2d6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
